In [1]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ================== FAST HOG + ExtraTrees (70/30,  aug) ==================
# Goal: run fast, keep decent accuracy. Optional quick tuning you can toggle.

import os, shutil, datetime, warnings, gc, time, json, joblib
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image

from joblib import Parallel, delayed
from skimage.feature import hog

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score, f1_score
from sklearn.preprocessing import label_binarize

In [3]:
# =============== PATHS (70/30 no-augmentation split) ===============
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
SPLIT70_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # <-- no-aug 70/30
TEST_DIR    = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

OUT_DIR = f"{DRIVE_ROOT}/FAST_ET_HOG_70_30_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(SPLIT70_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(SPLIT70_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)
print("✅ Copied 70/30 train/val + test to Colab local.")
print("OUT_DIR:", OUT_DIR)


✅ Copied 70/30 train/val + test to Colab local.
OUT_DIR: /content/drive/MyDrive/Colab Notebooks/FAST_ET_HOG_70_30_20250928_135054


In [4]:
# =================== SPEED-ORIENTED CONFIG ===================
RANDOM_STATE = 42
IMG_SIZE = (48, 48)


In [5]:
# HOG made lighter (fewer orientations / larger cell) → much faster
HOG_PARAMS = dict(
    orientations=8,
    pixels_per_cell=(10, 10),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    transform_sqrt=True,
    feature_vector=True
)

In [6]:
# FAST baseline model (ExtraTrees is typically faster than RF)
ET_PARAMS = dict(
    n_estimators=350,        # good sweet spot for speed vs accuracy
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

In [8]:
# Optional: very light/fast tuning (successive halving)
QUICK_TUNE = True   # <-- set True if you want a short tuning pass
TUNE_CV    = 2


In [9]:
# =================== UTILS ===================
def list_classes_from_dir(d):
    return sorted([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])

def load_paths_and_labels(root_dir, class_names):
    X_paths, y = [], []
    for c in class_names:
        d = os.path.join(root_dir, c)
        ps = sorted(glob(os.path.join(d, "*")))
        X_paths.extend(ps)
        y.extend([class_names.index(c)]*len(ps))
    return X_paths, np.array(y, dtype=np.int64)

def read_gray(path):
    img = Image.open(path).convert("L").resize(IMG_SIZE)
    return np.asarray(img, dtype=np.uint8)

def hog_one(path):
    return hog(read_gray(path), **HOG_PARAMS)

def compute_hog(paths, n_jobs=-1, batch_size=256):
    feats = Parallel(n_jobs=n_jobs, prefer="threads", batch_size=batch_size)(
        delayed(hog_one)(p) for p in paths
    )
    return np.vstack(feats)

def cache_path(split):
    return os.path.join(OUT_DIR, f"hog_{split}.npy")

def get_or_build_hog(split, paths):
    cp = cache_path(split)
    if os.path.exists(cp):
        print(f"⚡ Loading cached HOG: {cp}")
        return np.load(cp)
    print(f"🛠️ Building HOG for {split} …")
    X = compute_hog(paths, n_jobs=-1)
    np.save(cp, X)
    return X

def save_txt(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def plot_confusion(cm, classes, out_png, title):
    try:
        import seaborn as sns
        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)
    except Exception:
        plt.figure(figsize=(8,6))
        plt.imshow(cm, interpolation="nearest"); plt.colorbar()
        ticks = np.arange(len(classes))
        plt.xticks(ticks, classes, rotation=45); plt.yticks(ticks, classes)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i,j]), ha="center", va="center")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.close()

def plot_roc_ovr(y_true, y_proba, classes, out_png, title):
    y_bin = label_binarize(y_true, classes=list(range(len(classes))))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(classes)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(len(classes))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(classes)):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= len(classes)
    macro_auc = auc(all_fpr, mean_tpr)

    plt.figure(figsize=(8,6))
    for i in range(len(classes)):
        plt.plot(fpr[i], tpr[i], label=f"{classes[i]} (AUC={roc_auc[i]:.3f})")
    plt.plot(all_fpr, mean_tpr, linestyle="--", label=f"macro (AUC={macro_auc:.3f})")
    plt.plot([0,1],[0,1], linestyle=":", label="chance")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(title); plt.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.close()

In [10]:

# =================== DATA & FEATURES ===================
CLASSES = list_classes_from_dir(LOCAL_TRAIN)
print("Classes:", CLASSES)

train_paths, y_train = load_paths_and_labels(LOCAL_TRAIN, CLASSES)
val_paths,   y_val  = load_paths_and_labels(LOCAL_VAL,   CLASSES)
test_paths,  y_test = load_paths_and_labels(LOCAL_TEST,  CLASSES)
print(f"Counts: train={len(train_paths)}, val={len(val_paths)}, test={len(test_paths)}")

t0 = time.time()
X_train = get_or_build_hog("train", train_paths)
X_val   = get_or_build_hog("val",   val_paths)
X_test  = get_or_build_hog("test",  test_paths)
print(f"HOG total: {time.time()-t0:.1f}s")


Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Counts: train=28000, val=8616, test=7178
🛠️ Building HOG for train …
🛠️ Building HOG for val …
🛠️ Building HOG for test …
HOG total: 37.0s


In [11]:
# =================== FAST BASELINE: ExtraTrees ===================
et = ExtraTreesClassifier(**ET_PARAMS)
print("Training ExtraTrees (fast baseline)…")
t1 = time.time()
et.fit(X_train, y_train)
print(f"Fit time: {time.time()-t1:.1f}s")
joblib.dump(et, os.path.join(OUT_DIR, "et_hog_train_only.joblib"))


Training ExtraTrees (fast baseline)…
Fit time: 42.5s


['/content/drive/MyDrive/Colab Notebooks/FAST_ET_HOG_70_30_20250928_135054/et_hog_train_only.joblib']

In [12]:
# ----- VAL eval -----
y_val_pred = et.predict(X_val)
val_rep = classification_report(y_val, y_val_pred, target_names=CLASSES, digits=4)
save_txt(os.path.join(OUT_DIR, "val_report_BASELINE.txt"), val_rep)
cm_val = confusion_matrix(y_val, y_val_pred)
plot_confusion(cm_val, CLASSES, os.path.join(OUT_DIR, "val_confusion_BASELINE.png"),
               title="Confusion Matrix (VAL) – ExtraTrees+HOG")
if hasattr(et, "predict_proba"):
    plot_roc_ovr(y_val, et.predict_proba(X_val), CLASSES,
                 os.path.join(OUT_DIR, "val_roc_ovr_BASELINE.png"),
                 title="ROC (OvR) – VAL – ExtraTrees+HOG")

val_acc = accuracy_score(y_val, y_val_pred)
val_f1m = f1_score(y_val, y_val_pred, average="macro")
print(f"VAL BASELINE: acc={val_acc:.4f}, f1_macro={val_f1m:.4f}")

best_model, best_tag = et, "BASELINE"

VAL BASELINE: acc=0.4450, f1_macro=0.4019


In [14]:
# =================== OPTIONAL: VERY QUICK TUNING ===================
if QUICK_TUNE:
    print("🔎 Quick tuning with HalvingRandomSearchCV …")
    from sklearn.experimental import enable_halving_search_cv  # noqa: F401
    from sklearn.model_selection import HalvingRandomSearchCV

    param_dist = {
        "max_depth": [None, 24, 16],
        "min_samples_split": [2, 4],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", 0.5],
        "class_weight": ["balanced"],
        # "n_estimators": [150, 300, 450, 700],  # resource # Removed as it's the resource
    }
    search = HalvingRandomSearchCV(
        estimator=ExtraTreesClassifier(
            bootstrap=False, n_jobs=1, random_state=RANDOM_STATE
        ),
        param_distributions=param_dist,
        resource="n_estimators",
        min_resources=150,
        max_resources=700,
        factor=3,                 # bigger => faster
        scoring="f1_macro",
        cv=TUNE_CV,               # keep small for speed
        random_state=RANDOM_STATE,
        verbose=2,
        n_jobs=-1                 # parallelize outer only
    )
    t2 = time.time()
    search.fit(X_train, y_train)
    print(f"Tuning time: {time.time()-t2:.1f}s")
    tuned = search.best_estimator_
    print("Best (quick) params:", search.best_params_)

    y_val_pred_tuned = tuned.predict(X_val)
    val_acc_t = accuracy_score(y_val, y_val_pred_tuned)
    val_f1m_t = f1_score(y_val, y_val_pred_tuned, average="macro")
    print(f"VAL TUNED:    acc={val_acc_t:.4f}, f1_macro={val_f1m_t:.4f}")

    save_txt(os.path.join(OUT_DIR, "val_report_TUNED.txt"),
             classification_report(y_val, y_val_pred_tuned, target_names=CLASSES, digits=4))
    cm_val_t = confusion_matrix(y_val, y_val_pred_tuned)
    plot_confusion(cm_val_t, CLASSES, os.path.join(OUT_DIR, "val_confusion_TUNED.png"),
                   title="Confusion Matrix (VAL) – ExtraTrees+HOG (Tuned)")
    if hasattr(tuned, "predict_proba"):
        plot_roc_ovr(y_val, tuned.predict_proba(X_val), CLASSES,
                     os.path.join(OUT_DIR, "val_roc_ovr_TUNED.png"),
                     title="ROC (OvR) – VAL – ExtraTrees+HOG (Tuned)")

    if val_f1m_t >= val_f1m:
        best_model, best_tag = tuned, "TUNED"
        # =================== FINAL REFIT (TRAIN+VAL) & TEST ===================
X_trval = np.vstack([X_train, X_val])
y_trval = np.concatenate([y_train, y_val])

final_clf = best_model.__class__(**best_model.get_params())
# small bump to trees for final model if baseline chosen
if best_tag == "BASELINE" and isinstance(final_clf, ExtraTreesClassifier):
    final_clf.set_params(n_estimators=max(ET_PARAMS["n_estimators"], 400))
final_clf.set_params(n_jobs=-1)

print(f"Refitting ({best_tag}) on TRAIN+VAL …")
t3 = time.time()
final_clf.fit(X_trval, y_trval)
print(f"Refit time: {time.time()-t3:.1f}s")
joblib.dump(final_clf, os.path.join(OUT_DIR, f"et_hog_{best_tag}_final_trval.joblib"))

🔎 Quick tuning with HalvingRandomSearchCV …
n_iterations: 2
n_required_iterations: 2
n_possible_iterations: 2
min_resources_: 150
max_resources_: 700
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 4
n_resources: 150
Fitting 2 folds for each of 4 candidates, totalling 8 fits
----------
iter: 1
n_candidates: 2
n_resources: 450
Fitting 2 folds for each of 2 candidates, totalling 4 fits
Tuning time: 252.8s
Best (quick) params: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced', 'n_estimators': 450}
VAL TUNED:    acc=0.4493, f1_macro=0.4062
Refitting (TUNED) on TRAIN+VAL …
Refit time: 71.3s


['/content/drive/MyDrive/Colab Notebooks/FAST_ET_HOG_70_30_20250928_135054/et_hog_TUNED_final_trval.joblib']

In [15]:
# ----- TEST eval -----
y_test_pred = final_clf.predict(X_test)
test_report = classification_report(y_test, y_test_pred, target_names=CLASSES, digits=4)
save_txt(os.path.join(OUT_DIR, f"test_classification_report_{best_tag}.txt"), test_report)

cm_test = confusion_matrix(y_test, y_test_pred)
plot_confusion(cm_test, CLASSES, os.path.join(OUT_DIR, f"test_confusion_{best_tag}.png"),
               title=f"Confusion Matrix (TEST) – ExtraTrees+HOG ({best_tag})")
if hasattr(final_clf, "predict_proba"):
    plot_roc_ovr(y_test, final_clf.predict_proba(X_test), CLASSES,
                 os.path.join(OUT_DIR, f"test_roc_ovr_{best_tag}.png"),
                 title=f"ROC (OvR) – TEST – ExtraTrees+HOG ({best_tag})")

print("✅ Saved outputs to:", OUT_DIR)
print("\n==== VAL REPORT (BASELINE) ====\n", val_rep)
print("\n==== TEST REPORT (CHOSEN) ====\n", test_report)

# cleanup
del X_train, y_train, X_val, y_val, X_test, y_test
gc.collect()

✅ Saved outputs to: /content/drive/MyDrive/Colab Notebooks/FAST_ET_HOG_70_30_20250928_135054

==== VAL REPORT (BASELINE) ====
               precision    recall  f1-score   support

       angry     0.3459    0.3436    0.3448      1199
   disgusted     0.1811    0.5573    0.2734       131
     fearful     0.4110    0.2366    0.3003      1230
       happy     0.5843    0.6693    0.6239      2165
     neutral     0.3984    0.3409    0.3675      1490
         sad     0.3787    0.3168    0.3450      1449
   surprised     0.4766    0.6744    0.5585       952

    accuracy                         0.4450      8616
   macro avg     0.3966    0.4484    0.4019      8616
weighted avg     0.4416    0.4450    0.4350      8616


==== TEST REPORT (CHOSEN) ====
               precision    recall  f1-score   support

       angry     0.3774    0.3340    0.3544       958
   disgusted     0.3051    0.4865    0.3750       111
     fearful     0.5045    0.2715    0.3530      1024
       happy     0.5406   

25102